# ERA-14 — RAG Spike với bộ tài liệu NIST SP 800

Notebook kiểm chứng luồng RAG cốt lõi của dự án ở **Sprint 1** (trên 10 tài liệu PDF và 15 câu hỏi `v0`):
`10 file PDF → Chia đoạn & Embedding (process_document) → Lưu pgvector → Tìm Top-K (retrieve_top_k) → Gemini trả lời kèm trích dẫn [tài liệu, trang]`.

### Hướng dẫn chuẩn bị môi trường (chạy 1 lần trên terminal)
~~~bat
cd enterprise-research-agent
python -m venv .venv
.venv\Scripts\python -m pip install torch --index-url https://download.pytorch.org/whl/cu126
.venv\Scripts\python -m pip install -e "backend[ingest,llm,embeddings]" ipykernel
docker compose -p era-spike up -d --wait postgres redis
docker compose -p era-spike exec -T postgres psql -U era -d postgres -c "CREATE DATABASE era_spike_test;"
~~~
- Trong file `.env`: đặt `EMBEDDING_PROVIDER=local`, `LLM_PROVIDER=gemini`, `LLM_MODEL=gemini-3.5-flash-lite` và điền `GEMINI_API_KEY`.

## Bước 1. Nạp cấu hình `.env` và kết nối Database test (`era_spike_test`)

Đưa thư mục `backend/` vào `sys.path` để gọi được các hàm có sẵn trong repo, và trỏ kết nối sang database riêng `era_spike_test` trên `localhost`.

In [ ]:
import hashlib
import json
import os
import re
import sys
import time
from pathlib import Path

import torch

# 1. Tìm thư mục gốc enterprise-research-agent và chuyển working directory về đó để đọc đúng file .env
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    folder for folder in candidates
    if (folder / "backend/app/core/config.py").is_file()
)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))

from sqlalchemy import func, select, text
from sqlalchemy.engine import make_url
from app.core.config import Settings, get_settings
from app.db.session import get_engine, get_sessionmaker

# 2. Đọc .env và đổi tên database sang 'era_spike_test' (chỉ áp dụng trong tiến trình notebook)
base_settings = Settings(_env_file=PROJECT_ROOT / ".env")
test_db_url = make_url(base_settings.database_url).set(database="era_spike_test", host="localhost")
os.environ["DATABASE_URL"] = test_db_url.render_as_string(hide_password=False)

# Chọn kiểu Chunking có sẵn trong repo: "advanced" (LangChain Recursive) hoặc "baseline" (cắt cố định)
os.environ["CHUNK_STRATEGY"] = "advanced"

get_settings.cache_clear()
get_engine.cache_clear()
get_sessionmaker.cache_clear()
settings = get_settings()

CORPUS_DIR = PROJECT_ROOT / "eval/datasets/v0/corpus"  # đưa tập corpus gồm 10 file pdf vào đúng đường dẫn, xem ERA-13
QUESTIONS_PATH = PROJECT_ROOT / "eval/datasets/v0/questions.jsonl"
TOP_K = 5

# Tự động dùng GPU (cuda) nếu máy có card rời NVIDIA + PyTorch CUDA, ngược lại tự chạy bằng CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_NAME = f"cuda ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "cpu"

print("Database        :", test_db_url.database)
print("Device          :", DEVICE_NAME)
print("Chunk strategy  :", f"{settings.chunk_strategy} (size={settings.chunk_size}, overlap={settings.chunk_overlap})")
print("Embedding       :", f"{settings.embedding_provider} ({settings.local_embedding_model})")
print("Generation LLM  :", settings.llm_model)

## Bước 2. Tạo bảng PostgreSQL (`pgvector`) và đăng ký 10 file PDF vào DB

Vì các hàm `process_document()` và `retrieve_top_k()` của R2 làm việc trực tiếp với database, cell này sẽ:
1. Đọc 15 câu hỏi đánh giá bằng hàm `load_questions()` có sẵn của R2.
2. Tạo 5 bảng chuẩn (`users`, `collections`, `documents`, `chunks`, `research_jobs`) và bật extension `vector`.
3. Tạo 1 `User` test, 1 `Collection`, và đăng ký 10 file PDF trong `eval/datasets/v0/corpus/` vào bảng `documents`.

In [ ]:
from app.core.security import hash_password
from app.models import Base, Chunk, Collection, Document, DocumentStatus, User
from workers.research.retrieve import load_questions

# 1. Đọc danh sách 15 câu hỏi và 10 file PDF
questions = load_questions(QUESTIONS_PATH)
pdf_paths = sorted(CORPUS_DIR.glob("*.pdf"))
print(f"Đã tìm thấy {len(questions)} câu hỏi và {len(pdf_paths)} file PDF.")

# 2. Bật extension pgvector và tạo bảng trong DB test
engine = get_engine()
with engine.begin() as conn:
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector"))
    Base.metadata.create_all(conn)

# 3. Tạo User, Collection và 10 bản ghi Document (nếu chưa có)
collection_name = f"NIST Corpus v0 ({settings.chunk_strategy})"
document_ids = {}

with get_sessionmaker().begin() as db:
    user = db.scalar(select(User).where(User.email == "era-spike@example.test"))
    if user is None:
        user = User(email="era-spike@example.test", password_hash=hash_password("spike-pass-123"))
        db.add(user)
        db.flush()

    collection = db.scalar(
        select(Collection).where(Collection.owner_id == user.id, Collection.name == collection_name)
    )
    if collection is None:
        collection = Collection(owner_id=user.id, name=collection_name)
        db.add(collection)
        db.flush()

    owner_id = user.id
    collection_id = collection.id

    for path in pdf_paths:
        file_sha256 = hashlib.sha256(path.read_bytes()).hexdigest()
        doc = db.scalar(
            select(Document).where(Document.collection_id == collection_id, Document.filename == path.name)
        )
        if doc is None:
            doc = Document(
                owner_id=owner_id,
                collection_id=collection_id,
                filename=path.name,
                content_type="application/pdf",
                size_bytes=path.stat().st_size,
                sha256=file_sha256,
                storage_path=str(path.resolve()),
                status=DocumentStatus.QUEUED,
            )
            db.add(doc)
            db.flush()
        document_ids[path.name] = doc.id

print(f"Đã chuẩn bị Collection '{collection_name}' với {len(document_ids)} tài liệu PDF.")

## Bước 3. Chạy Ingestion 1 lượt cho cả 10 file PDF (`process_document` của R2)

Gọi trực tiếp hàm **`process_document(document_id)` có sẵn của R2** cho từng file PDF:
- Hàm tự động đọc PDF theo từng trang → cắt chunk → nhúng thành vector 384 chiều → lưu vào bảng `chunks` (`pgvector`) và đổi trạng thái sang `READY`.
- Nếu tài liệu đã `READY` từ lần chạy trước, `process_document()` sẽ tự động bỏ qua (idempotent).

In [ ]:
from workers.embedding import _local_model
from workers.ingestion.tasks import process_document

# Nạp model embedding local lên GPU (nếu có) hoặc CPU
if settings.embedding_provider == "local":
    _local_model().to(DEVICE)

total_chunks = 0
for filename, doc_id in document_ids.items():
    t0 = time.perf_counter()
    process_document(str(doc_id))  # Hàm ingestion có sẵn của R2
    elapsed = time.perf_counter() - t0

    with get_sessionmaker()() as db:
        doc = db.get(Document, doc_id)
        chunk_count = db.scalar(select(func.count()).select_from(Chunk).where(Chunk.document_id == doc_id))

    total_chunks += chunk_count
    print(f"[{doc.status.value.upper():5s}] {filename:22s} | {chunk_count:4d} chunks | {elapsed:.2f}s", flush=True)

print(f"\nHoàn tất: Tổng cộng {total_chunks} chunks đã có mặt trong PostgreSQL (pgvector).")

## Bước 4. Kiểm tra Retrieval Top-5 (`retrieve_top_k` & `run_dataset` của R2)

1. Gọi `retrieve_top_k()` cho câu hỏi `q01` để xem 5 đoạn văn (`chunks`) có điểm tương đồng cao nhất lấy ra từ `pgvector`.
2. Gọi `run_dataset()` để đo tỷ lệ tìm trúng tài liệu nguồn (`Source Hit@5`) trên toàn bộ 15 câu hỏi.

In [ ]:
from workers.research.retrieve import retrieve_top_k, run_dataset

# 1. Xem thử Top-5 chunks tìm được cho câu hỏi q01
q1 = questions[0]
sample_chunks = retrieve_top_k(
    question=q1["question"],
    collection_id=collection_id,
    owner_id=owner_id,
    top_k=TOP_K,
)
print(f"Câu hỏi ({q1['id']}): {q1['question']}\n")
for c in sample_chunks:
    print(f"  Rank #{c['rank']} | {c['filename']} (Trang {c['page']}) | score={c['score']:.4f}")
    print(f"  -> {c['text'][:200]}...\n")

# 2. Chạy đánh giá Retrieval Hit@5 trên toàn bộ 15 câu hỏi
retrieval_eval = run_dataset(
    questions=questions,
    collection_id=collection_id,
    owner_id=owner_id,
    top_k=TOP_K,
)
retrieval_by_id = {item["question_id"]: item for item in retrieval_eval["questions"]}
print("=== KẾT QUẢ RETRIEVAL 15 CÂU HỎI ===")
print(json.dumps(retrieval_eval["metrics"], indent=2))

## Bước 5. Hàm gọi Gemini LLM sinh câu trả lời kèm trích dẫn (`Phần của R3`)

Hàm `answer_with_citations(question, chunks)` thực hiện 3 việc đơn giản:
1. **Đánh số nguồn `[1]..[5]`** cho 5 đoạn chunks lấy được từ bước Retrieval.
2. **Gửi câu hỏi + 5 đoạn nguồn vào Gemini (`gemini-3.5-flash-lite`)**, yêu cầu trả lời ngắn gọn, chỉ dựa vào nguồn và gắn dấu `[1]`, `[2]` sau mỗi ý.
3. **Bóc tách trích dẫn:** Dùng regex `\[\d+\]` tìm các dấu `[1]`, `[2]` xuất hiện trong câu trả lời để gắn lại đúng tên file PDF (`filename`) và số trang (`page`).

In [ ]:
from google import genai
from google.genai import types

def answer_with_citations(question: str, chunks: list[dict]) -> dict:
    """Gọi Gemini sinh câu trả lời từ các chunks và trích xuất danh sách tài liệu/trang được trích dẫn."""
    if not chunks:
        return {"answer": "Không tìm thấy đoạn tài liệu nguồn nào phù hợp.", "citations": []}

    # 1. Đánh số [1], [2]... cho từng đoạn chunk
    sources = {f"[{i}]": chunk for i, chunk in enumerate(chunks, start=1)}
    context_text = "\n\n".join(
        f"{marker} Tài liệu: {c['filename']} (Trang {c['page']})\n{c['text']}"
        for marker, c in sources.items()
    )

    # 2. Tạo prompt và gọi Gemini API (có timeout 30s và tự retry khi gặp lỗi 429)
    system_instruction = (
        "Bạn là trợ lý nghiên cứu tài liệu. Hãy trả lời câu hỏi bằng tiếng Việt ngắn gọn, rõ ràng, "
        "chỉ dựa trên các đoạn nguồn bên dưới. "
        "Ngay sau mỗi thông tin lấy từ đoạn nguồn nào, bắt buộc gắn ký hiệu trích dẫn của đoạn đó (ví dụ [1], [2]). "
        "Chỉ sử dụng các ký hiệu [1]..[5] đã có trong nguồn."
    )
    prompt = f"Câu hỏi: {question}\n\nCác đoạn nguồn:\n{context_text}"

    api_key = settings.gemini_api_key.get_secret_value() if settings.gemini_api_key else ""
    if not api_key:
        raise ValueError("Chưa điền GEMINI_API_KEY trong file .env!")

    with genai.Client(
        api_key=api_key,
        http_options=types.HttpOptions(
            timeout=30_000,
            retry_options=types.HttpRetryOptions(
                attempts=3,
                http_status_codes=[429, 503],
                initial_delay=10.0,
                max_delay=35.0,
            ),
        ),
    ) as client:
        response = client.models.generate_content(
            model=settings.llm_model,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0,
            ),
        )

    answer_text = (response.text or "").strip()

    # 3. Tìm các marker [1], [2] hoặc [1, 2] đã dùng trong câu trả lời và map về file PDF + số trang
    raw_brackets = re.findall(r"\[(\d+(?:\s*,\s*\d+)*)\]", answer_text)
    used_markers = list(
        dict.fromkeys(f"[{num.strip()}]" for group in raw_brackets for num in group.split(","))
    )
    citations = []
    for marker in used_markers:
        if marker in sources:
            c = sources[marker]
            citations.append({
                "marker": marker,
                "document_id": c["document_id"],
                "chunk_id": c["chunk_id"],
                "filename": c["filename"],
                "page": c["page"],
                "text_preview": c["text"][:250] + "...",
            })

    return {"answer": answer_text, "citations": citations}

## Bước 6. Chạy Gemini trả lời toàn bộ 15 câu hỏi, Kiểm tra trích dẫn & Lưu file kết quả

Chạy hết cả **15 câu hỏi (`q01` → `q15`)**, in ra câu trả lời kèm trích dẫn để đối chiếu và tự động lưu toàn bộ kết quả vào file `eval/results/rag_spike_results.json`.

In [ ]:
# Chạy toàn bộ 15 câu hỏi từ q01 đến q15 và lưu kết quả
TEST_IDS = [q["id"] for q in questions]
spike_answers = []

for qid in TEST_IDS:
    item = retrieval_by_id[qid]
    t0 = time.perf_counter()
    result = answer_with_citations(item["question"], item["results"])
    elapsed = round(time.perf_counter() - t0, 2)

    spike_answers.append({
        "question_id": qid,
        "question": item["question"],
        "answer": result["answer"],
        "citations": result["citations"],
        "generation_seconds": elapsed,
    })

    print("=" * 90)
    print(f"CÂU HỎI ({qid} - {elapsed:.2f}s): {item['question']}")
    print("-" * 90)
    print("CÂU TRẢ LỜI:")
    print(result["answer"])
    print("\nTRÍCH DẪN NGUỒN (ĐỐI CHIẾU VỚI PDF):")
    for cit in result["citations"]:
        print(f"  {cit['marker']} {cit['filename']} - Trang {cit['page']}:")
        print(f"      \"{cit['text_preview']}\"")
    print()

# Lưu toàn bộ kết quả ra file eval/results/rag_spike_results.json
out_path = PROJECT_ROOT / "eval/results/rag_spike_results.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(
    json.dumps(
        {
            "chunk_strategy": settings.chunk_strategy,
            "embedding_model": settings.local_embedding_model,
            "llm_model": settings.llm_model,
            "retrieval_metrics": retrieval_eval["metrics"],
            "results": spike_answers,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print(f"Đã lưu trọn bộ kết quả {len(spike_answers)} câu hỏi tại: {out_path.relative_to(PROJECT_ROOT)}")